# Getting our links right

For the polars library, we can crawl the links in the index page. We can then use the links to download the data.

In [2]:
import requests
from bs4 import BeautifulSoup

def fetch_html_content(url):
    """
    Fetches the HTML content from the given URL.

    Args:
        url (str): The URL to fetch the HTML content from.

    Returns:
        str: The HTML content of the URL.

    Raises:
        requests.HTTPError: If there is an error while fetching the content (e.g., 404).
    """
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def extract_links(html_content, base_url):
    """
    Extracts links from the given HTML content and converts relative links to absolute links.

    Args:
        html_content (str): The HTML content to extract links from.
        base_url (str): The base URL to use for converting relative links.

    Returns:
        list: A list of absolute links.

    """
    soup = BeautifulSoup(html_content, "html.parser")
    all_links = [a["href"] for a in soup.find_all("a", href=True)]
    new_links = []
    for link in all_links:
        if link.startswith("http"):
            # remove external links
            pass
        else:
            # add base URL to relative links
            new_links.append(base_url + link)

    # Drop duplicates
    new_links = list(set(new_links))
    return new_links

# Usage example
url = "https://docs.pola.rs/py-polars/html/reference/index.html"  
base_url = "https://docs.pola.rs/py-polars/html/reference/"  

# Fetch the HTML content
html_content = fetch_html_content(url)

# Extract links
links = extract_links(html_content, base_url)

# The WebBaseLoader

In langchain, we have a WebBaseLoader that can be used to download data from the web. We can use this to download the data from the links we have crawled.

In [3]:
import duckdb
from langchain.document_loaders import WebBaseLoader
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores.duckdb import DuckDB

# Load Polars documentation (replace with actual URL)
loader = WebBaseLoader(links[:30])
docs = loader.load()




# Shhh... It's a secret

You will need to set you Open AI API key in the `OPENAI_API_KEY` environment variable. You can get your API key from the Open AI website.

In [ ]:
import os
# set enviorn variable OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = ""

# Taxonomy of the polars library

We will be using a very simple taxonomy for the polars library. We will be using the following categories:

- "topic": The topic of the article
- "functionality": The functionality of the method or class covered in the article

In [5]:
# Define schema for Polars documentation tags (same as before)
schema = {
    "properties": {
        "topic": {"type": "string", "enum": ["DataFrame", "Series", "LazyFrame", "Other"]},
        "functionality": {"type": "string", "enum": ["Manipulation", "Aggregation", "Query", "Other"]}
    },
    "required": ["topic"]
}


# Tagging the data

We will be using the Open AI API to tag the data. We will be using the new `gpt-4o` model for this purpose. A smaller model would probably work just as well but pay attention to context length!

In [6]:
from langchain_community.document_transformers.openai_functions import (
    create_metadata_tagger,
)
from langchain_openai import ChatOpenAI

# Initialize OpenAI LLM and metadata tagger
llm = ChatOpenAI(temperature=0, model="gpt-4o")
tagger = create_metadata_tagger(metadata_schema=schema, llm=llm)

# Tag the documents
tagged_docs = tagger.transform_documents(docs)



In [13]:
for doc in tagged_docs:
    doc.metadata['version'] = '0.20'
    doc.metadata['date'] = '20xx-xx-xx'
    doc.metadata['tagging_llm'] = 'gpt-4o'
    doc.metadata['taxonomy_version'] = '0.1'

In [15]:
tagged_docs[0].metadata

{'topic': 'DataFrame',
 'functionality': 'Aggregation',
 'source': 'https://docs.pola.rs/py-polars/html/reference/expressions/api/polars.mean.html',
 'title': 'polars.mean — Polars  documentation',
 'language': 'en',
 'version': '0.20',
 'date': '20xx-xx-xx',
 'tagging_llm': 'gpt-4o',
 'taxonomy_version': '0.1'}

# Putting it into a Database

We will be using DuckDB as a lightweight vector database to store the tagged data.

For production use, you would probably want to use a more robust database. 

## If we have it in a dataframe...


For Pandas dataframes, we can use the `DataFrameLoader` to load the data as the docouments type
For Polars we can use `PolarsDataFrameLoader`


```python
from langchain_community.document_loaders import DataFrameLoader # For Pandas
from langchain_community.document_loaders import PolarsDataFrameLoader # For Polars

loader = DataFrameLoader(df, page_content_column="Page Content")
```

In [14]:
# Create DuckDB connection and initialize vector store
conn = duckdb.connect()
embedding_function = OpenAIEmbeddings()

vector_store = DuckDB.from_documents(tagged_docs, embedding_function)


# Example query with metadata filtering
results = vector_store.similarity_search(
    "Polars DataFrame operations", filter={"topic": "DataFrame"}
)

# Display results (first few for brevity)
for doc in results[:3]:
    print(doc.page_content[:100], doc.metadata)

/opt/homebrew/lib/python3.11/site-packages/langchain_community/vectorstores/duckdb.py:105: UserWarning: No DuckDB connection provided. A new connection will be created.This connection is running in memory and no data will be persisted.To persist data, specify `connection=duckdb.connect(...)` when using the API. Please review the documentation of the vectorstore for security recommendations on configuring the connection.
  warnings.warn(










Aggregation — Polars  documentation


































Skip to main content

 {'topic': 'LazyFrame', 'functionality': 'Aggregation', 'source': 'https://docs.pola.rs/py-polars/html/reference/lazyframe/aggregation.html', 'title': 'Aggregation — Polars  documentation', 'language': 'en', 'version': '0.20', 'date': '20xx-xx-xx', 'tagging_llm': 'gpt-4o', 'taxonomy_version': '0.1'}








Data types — Polars  documentation


































Skip to main content


 {'topic': 'Other', 'source': 'https://docs.pola.rs/py-polars/html/reference/datatypes.html#numeric', 'title': 'Data types — Polars  documentation', 'language': 'en', 'version': '0.20', 'date': '20xx-xx-xx', 'tagging_llm': 'gpt-4o', 'taxonomy_version': '0.1'}








Input/output — Polars  documentation


































Skip to main content
 {'topic': 'DataFrame', 'functionality': 'Other', 'source': 'https://docs.pola.rs/py-polars/html/reference/io.html#parquet', 'title': 'Inpu